# Visualize Raw DNS Snapshot (with Interpolation)

This notebook loads a single, unprocessed 3D snapshot from a raw binary file, interpolates 2D slices onto a uniform grid, and visualizes them.

**Workflow:**
1.  Set all parameters in the **User Configuration** cell.
2.  Run the **Load Snapshot Data** cell once to load the binary file into memory.
3.  Modify slice indices in the parameter cells for each plot type and run the individual plot cells as needed.

### 1. Setup and Imports

In [ ]:
import numpy as np
from pathlib import Path
import logging

# Import the new plotting functions from the refactored package
from mhd_surrogate_core.plotting import (
    plot_interpolated_xz_slice,
    plot_interpolated_xy_slice,
    plot_interpolated_yz_slice
)

### 2. User Configuration

**Action Required:** Set your parameters in this cell. You can re-run this cell to apply changes without reloading the data.

In [ ]:
# <<< 1. SET SNAPSHOT FILE PATH AND PARAMETERS >>>
snapshot_file_path = Path("/raid/skowronek/preprocessed_dns_output/01-Cold_Runs/01-Re16K_Ha325/raw/patt3d_vx3d_000609")
nx, ny, nz = 2301, 481, 121  # Grid dimensions
source_channel_labels = ['vx', 'vy', 'vz', 'T'] # IMPORTANT: Must be in the order they appear in the file

# <<< 2. DEFINE CHANNEL NAME MAPPING >>>
channel_map = {
    'vx': 'u',
    'vy': 'v',
    'vz': 'w'
}

# <<< 3. SET INTERPOLATION GRID SIZE >>>
# Number of points for the new uniform y-grid and z-grid
num_interp_points_y = 1024
num_interp_points_z = 1024

# <<< 4. SET GLOBAL PLOT SIZING AND UNIT PARAMETERS >>>
global_base_size = 20
global_min_size = 0.1
global_unit_label = "m/s"

print("Configuration set. You can now run the 'Load Snapshot Data' cell.")

### 3. Load Snapshot Data

**Run this cell only once** to load the raw binary data from the specified file.

In [ ]:
# --- Data Loading and Initialization ---
snapshot_data_3d = None
raw_coords = {}

if not snapshot_file_path.exists():
    logging.error(f"ERROR: Snapshot file not found at {snapshot_file_path}. Please set the correct path in the configuration cell.")
else:
    try:
        input_dtype = np.float64
        with open(snapshot_file_path, 'rb') as f:
            # 1. Read coordinates
            x_coords_raw = np.fromfile(f, dtype=input_dtype, count=nx)
            y_coords_raw = np.fromfile(f, dtype=input_dtype, count=ny)
            z_coords_raw = np.fromfile(f, dtype=input_dtype, count=nz)
            
            # 2. Read all channel data
            channel_data_1d = np.fromfile(f, dtype=input_dtype)
        
        # 3. Reshape and transpose the data to a logical (x, y, z, channel) layout
        num_input_channels = len(source_channel_labels)
        # Original physical layout is (z, chan, y, x)
        data_4d_physical = channel_data_1d.reshape((nz, num_input_channels, ny, nx))
        
        # <<< FIX: Correctly transpose from (z, chan, y, x) to (x, y, z, chan) >>>
        snapshot_data_3d = data_4d_physical.transpose(3, 2, 0, 1).astype(np.float32)
        
        raw_coords = {
            'labels': source_channel_labels,
            'x': x_coords_raw,
            'y': y_coords_raw,
            'z': z_coords_raw,
        }
        print("Snapshot data loaded successfully into memory.")
        print(f"Final data shape: {snapshot_data_3d.shape}")
    except Exception as e:
        logging.error(f"Failed to load or process snapshot file: {e}")
        logging.error("Please check grid dimensions (nx, ny, nz) and source channel labels.")

# Identify velocity components to be plotted
velocity_components = sorted([c for c in raw_coords.get('labels', []) if c in channel_map])
if not velocity_components:
    logging.warning("Warning: No velocity components matching the channel_map found.")

---

### 4. Spatial Slice Plots (2D)

#### 4.1 X-Z Slices (at constant Y)

In [ ]:
# === Parameters and Color Scale for X-Z Slices ===
# <<< MODIFY THESE VALUES >>>
plot_y_index_xz = ny // 2
# ---

vmin_xz, vmax_xz = None, None
if snapshot_data_3d is not None and velocity_components:
    slices = []
    for vc in velocity_components:
        vc_idx = raw_coords['labels'].index(vc)
        slices.append(snapshot_data_3d[:, plot_y_index_xz, :, vc_idx])
    
    vmin_xz = min(s.min() for s in slices)
    vmax_xz = max(s.max() for s in slices)
    print(f"Global color scale for X-Z slices (y={plot_y_index_xz}): [{vmin_xz:.3f}, {vmax_xz:.3f}]")

In [ ]:
# === Plot X-Z Slice for u (vx) ===
if 'vx' in velocity_components:
    plot_interpolated_xz_slice(
        snapshot_data_3d=snapshot_data_3d,
        raw_coords=raw_coords,
        channel='vx',
        y_index=plot_y_index_xz,
        num_interp_points_z=num_interp_points_z,
        channel_alias=channel_map.get('vx'),
        vmin=vmin_xz,
        vmax=vmax_xz,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )

In [ ]:
# === Plot X-Z Slice for v (vy) ===
if 'vy' in velocity_components:
    plot_interpolated_xz_slice(
        snapshot_data_3d=snapshot_data_3d,
        raw_coords=raw_coords,
        channel='vy',
        y_index=plot_y_index_xz,
        num_interp_points_z=num_interp_points_z,
        channel_alias=channel_map.get('vy'),
        vmin=vmin_xz,
        vmax=vmax_xz,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )

In [ ]:
# === Plot X-Z Slice for w (vz) ===
if 'vz' in velocity_components:
    plot_interpolated_xz_slice(
        snapshot_data_3d=snapshot_data_3d,
        raw_coords=raw_coords,
        channel='vz',
        y_index=plot_y_index_xz,
        num_interp_points_z=num_interp_points_z,
        channel_alias=channel_map.get('vz'),
        vmin=vmin_xz,
        vmax=vmax_xz,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )

#### 4.2 X-Y Slices (at constant Z)

In [ ]:
# === Parameters and Color Scale for X-Y Slices ===
# <<< MODIFY THESE VALUES >>>
plot_z_index_xy = nz // 2
# ---

vmin_xy, vmax_xy = None, None
if snapshot_data_3d is not None and velocity_components:
    slices = []
    for vc in velocity_components:
        vc_idx = raw_coords['labels'].index(vc)
        slices.append(snapshot_data_3d[:, :, plot_z_index_xy, vc_idx])
    
    vmin_xy = min(s.min() for s in slices)
    vmax_xy = max(s.max() for s in slices)
    print(f"Global color scale for X-Y slices (z={plot_z_index_xy}): [{vmin_xy:.3f}, {vmax_xy:.3f}]")

In [ ]:
# === Plot X-Y Slice for u (vx) ===
if 'vx' in velocity_components:
    plot_interpolated_xy_slice(
        snapshot_data_3d=snapshot_data_3d,
        raw_coords=raw_coords,
        channel='vx',
        z_index=plot_z_index_xy,
        num_interp_points_y=num_interp_points_y,
        channel_alias=channel_map.get('vx'),
        vmin=vmin_xy,
        vmax=vmax_xy,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )

In [ ]:
# === Plot X-Y Slice for v (vy) ===
if 'vy' in velocity_components:
    plot_interpolated_xy_slice(
        snapshot_data_3d=snapshot_data_3d,
        raw_coords=raw_coords,
        channel='vy',
        z_index=plot_z_index_xy,
        num_interp_points_y=num_interp_points_y,
        channel_alias=channel_map.get('vy'),
        vmin=vmin_xy,
        vmax=vmax_xy,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )

In [ ]:
# === Plot X-Y Slice for w (vz) ===
if 'vz' in velocity_components:
    plot_interpolated_xy_slice(
        snapshot_data_3d=snapshot_data_3d,
        raw_coords=raw_coords,
        channel='vz',
        z_index=plot_z_index_xy,
        num_interp_points_y=num_interp_points_y,
        channel_alias=channel_map.get('vz'),
        vmin=vmin_xy,
        vmax=vmax_xy,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )

#### 4.3 Y-Z Slices (at constant X)

In [ ]:
# === Parameters and Color Scale for Y-Z Slices ===
# <<< MODIFY THESE VALUES >>>
plot_x_index_yz = nx // 2
# ---

vmin_yz, vmax_yz = None, None
if snapshot_data_3d is not None and velocity_components:
    slices = []
    for vc in velocity_components:
        vc_idx = raw_coords['labels'].index(vc)
        slices.append(snapshot_data_3d[plot_x_index_yz, :, :, vc_idx])
    
    vmin_yz = min(s.min() for s in slices)
    vmax_yz = max(s.max() for s in slices)
    print(f"Global color scale for Y-Z slices (x={plot_x_index_yz}): [{vmin_yz:.3f}, {vmax_yz:.3f}]")

In [ ]:
# === Plot Y-Z Slice for u (vx) ===
if 'vx' in velocity_components:
    plot_interpolated_yz_slice(
        snapshot_data_3d=snapshot_data_3d,
        raw_coords=raw_coords,
        channel='vx',
        x_index=plot_x_index_yz,
        num_interp_points_y=num_interp_points_y,
        num_interp_points_z=num_interp_points_z,
        channel_alias=channel_map.get('vx'),
        vmin=vmin_yz,
        vmax=vmax_yz,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )

In [ ]:
# === Plot Y-Z Slice for v (vy) ===
if 'vy' in velocity_components:
    plot_interpolated_yz_slice(
        snapshot_data_3d=snapshot_data_3d,
        raw_coords=raw_coords,
        channel='vy',
        x_index=plot_x_index_yz,
        num_interp_points_y=num_interp_points_y,
        num_interp_points_z=num_interp_points_z,
        channel_alias=channel_map.get('vy'),
        vmin=vmin_yz,
        vmax=vmax_yz,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )

In [ ]:
# === Plot Y-Z Slice for w (vz) ===
if 'vz' in velocity_components:
    plot_interpolated_yz_slice(
        snapshot_data_3d=snapshot_data_3d,
        raw_coords=raw_coords,
        channel='vz',
        x_index=plot_x_index_yz,
        num_interp_points_y=num_interp_points_y,
        num_interp_points_z=num_interp_points_z,
        channel_alias=channel_map.get('vz'),
        vmin=vmin_yz,
        vmax=vmax_yz,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )